# Thesis - XAI
by Azhar Rafiq

Dataset:
- ✅[ downloaded] RSNA (kaggle.com/competitions/rsna-intracranial-hemorrhage-detection)
- [later] PhysioNet (physionet.org/content/ct-ich/1.3.1/)

In [1]:
#library
import os
from dotenv import load_dotenv

load_dotenv()

import sys
import glob

import numpy as np
import pandas as pd

import seaborn as sns
sns.set_style('darkgrid')
import matplotlib.pyplot as plt

import random

import tensorflow as tf
from tensorflow.keras import models, layers, losses, optimizers
from tensorflow.keras import callbacks

import joblib

#

import pydicom
from pathlib import Path

print("✅ Libraries imported succesfully\n")

print(f"🐍 Python Version: {sys.version.split()[0]}")
print(f"📖 Pandas Version: {pd.__version__}")
print(f"📖 Numpy Version: {np.__version__}")
print(f"📖 Seaborn Version: {sns.__version__}")

print(f"📖 Tensorflow Version: {tf.__version__}")
print(f"📖 Joblib Version: {joblib.__version__}")

print(f"📖 pydicom Version: {pydicom.__version__}")


I0000 00:00:1778729725.183793 2029015 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1778729725.201873 2029015 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1778729727.444779 2029015 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1778729752.604877 2029015 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.

✅ Libraries imported succesfully

🐍 Python Version: 3.12.3
📖 Pandas Version: 3.0.2
📖 Numpy Version: 2.4.4
📖 Seaborn Version: 0.13.2
📖 Tensorflow Version: 2.21.0
📖 Joblib Version: 1.5.3
📖 pydicom Version: 3.0.2


In [2]:
# #import RSNA dataset

# import kagglehub
# kagglehub.login()


In [3]:
# #specify dataset directory
# os.environ['KAGGLEHUB_CACHE'] = os.getenv('KAGGLE_CACHE')
# os.makedirs(os.environ['KAGGLEHUB_CACHE'], exist_ok=True)

# rsnapath = kagglehub.competition_download('rsna-intracranial-hemorrhage-detection')
# rsnapath
# #files already downloaded

In [4]:
#load files
#files already separated by the RSNA Kaggle Challenge

BASE = Path('.')
RSNA_TRAIN_DIR = BASE / '/rds/projects/k/karwatha-karwath-hds-pg-research/axr1222/data/competitions/rsna-intracranial-hemorrhage-detection/rsna-intracranial-hemorrhage-detection/stage_2_train'
RSNA_TEST_DIR = BASE / '/rds/projects/k/karwatha-karwath-hds-pg-research/axr1222/data/competitions/rsna-intracranial-hemorrhage-detection/rsna-intracranial-hemorrhage-detection/stage_2_test'

#count RSNA training files
print(f'RSNA Train DICOMs: {len(list(RSNA_TRAIN_DIR.glob("*.dcm")))}')
print(f'RSNA Test DICOMs: {len(list(RSNA_TEST_DIR.glob("*.dcm")))}')

RSNA Train DICOMs: 752803
RSNA Test DICOMs: 121142


In [5]:

print(tf.config.list_physical_devices('GPU'))

[]


E0000 00:00:1778729770.672026 2029015 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [6]:
#seed
SEED = 20260605
def change_seed(SEED=42):
    #lock python and libraries random generator to make sure reproducibility
    os.environ['PYTHONHASHSEED'] = str(SEED)
    random.seed(SEED)
    np.random.seed(SEED)
    tf.random.set_seed(SEED)
change_seed(SEED) #using the assigned seed